# Insurance Claims Analysis & Fraud Detection

## Introduction

Lorsqu'un conducteur est impliqué dans un accident, il dépose une déclaration de sinistre auprès de sa compagnie d'assurance. Cette déclaration décrit les circonstances de l'accident, les dommages subis et les montants réclamés. La grande majorité de ces déclarations sont légitimes.

Cependant, une partie des déclarations peut être frauduleuse : accidents simulés, dommages exagérés, circonstances inventées. La fraude représente un coût significatif pour les compagnies d'assurance, qui se répercute ensuite sur l'ensemble des assurés sous forme de primes plus élevées.

L'analyse des données peut aider à identifier les déclarations qui présentent des caractéristiques inhabituelles — non pas pour accuser automatiquement un client, mais pour prioriser les dossiers qui méritent une vérification manuelle approfondie.

Dans ce projet, je vais :
- explorer les caractéristiques des déclarations de sinistres ;
- analyser les profils associés aux déclarations frauduleuses ;
- construire et comparer plusieurs modèles de Machine Learning capables de prédire le risque de fraude ;
- interpréter les résultats dans une perspective métier.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve
)

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
Path('figures').mkdir(exist_ok=True)

## Étape 1 — Chargement et présentation du dataset

In [ ]:
# Je charge le dataset depuis le dossier courant.
df = pd.read_csv('insurance_claims.csv')

print(f"Dimensions : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
print(f"\nColonnes disponibles :")
print(df.columns.tolist())

In [ ]:
# Je regarde les premières lignes pour comprendre comment les données sont organisées.
df.head()

In [ ]:
# Informations générales sur les types de données.
df.info()

In [ ]:
# Statistiques descriptives des variables numériques.
df.describe().round(2)

In [ ]:
# Je vérifie les valeurs manquantes et les doublons.
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

print("Valeurs manquantes :")
print(missing[missing > 0].to_frame('count').assign(pct=missing_pct[missing > 0]))

print(f"\nDoublons : {df.duplicated().sum()}")

In [ ]:
# Je vérifie si le dataset contient des valeurs spéciales comme '?' qui masquent des manquants.
for col in df.select_dtypes(include='object').columns:
    n = (df[col] == '?').sum()
    if n > 0:
        print(f"{col} : {n} valeurs '?'")

## Étape 2 — Identification des groupes de variables

Je regroupe les colonnes par catégorie pour mieux comprendre la structure du dataset.

In [ ]:
# Je classe les variables par groupe selon leur rôle probable.
all_cols = df.columns.tolist()

client_vars = [c for c in all_cols if any(k in c.lower() for k in
    ['age', 'insured', 'gender', 'education', 'occupation', 'marital', 'relationship'])]

policy_vars = [c for c in all_cols if any(k in c.lower() for k in
    ['policy', 'premium', 'deductible', 'coverage', 'state', 'bind'])]

incident_vars = [c for c in all_cols if any(k in c.lower() for k in
    ['incident', 'collision', 'severity', 'witnesses', 'authorities', 'hour', 'date'])]

vehicle_vars = [c for c in all_cols if any(k in c.lower() for k in
    ['vehicle', 'make', 'model', 'year', 'auto'])]

financial_vars = [c for c in all_cols if any(k in c.lower() for k in
    ['claim', 'capital', 'loss', 'injury', 'property', 'vehicle_claim', 'total'])]

print("Informations client        :", client_vars)
print("\nInformations police        :", policy_vars)
print("\nInformations incident      :", incident_vars)
print("\nInformations véhicule      :", vehicle_vars)
print("\nInformations financières   :", financial_vars)

## Étape 3 — Nettoyage des données

In [ ]:
# Je remplace les '?' par NaN pour qu'ils soient traités comme des valeurs manquantes.
df.replace('?', np.nan, inplace=True)

# Je vérifie les colonnes à identifier — un identifiant ne doit pas entrer dans le modèle.
id_like = [c for c in df.columns if 'id' in c.lower() or 'number' in c.lower()]
print("Colonnes potentiellement identifiantes :", id_like)

In [ ]:
# Je supprime les colonnes identifiantes qui n'apportent pas d'information
# sur le risque de fraude et pourraient perturber les modèles.
cols_to_drop = [c for c in id_like if c in df.columns]

# Je retire également les colonnes avec plus de 50% de valeurs manquantes
# car elles ne seraient pas fiables dans un modèle.
high_missing = [c for c in df.columns if df[c].isnull().mean() > 0.5]

cols_to_drop = list(set(cols_to_drop + high_missing))
print("Colonnes retirées :", cols_to_drop)

df_clean = df.drop(columns=cols_to_drop, errors='ignore').copy()
print(f"\nDataset après nettoyage : {df_clean.shape[0]:,} lignes × {df_clean.shape[1]} colonnes")

## Étape 4 — Analyse de la variable cible

In [ ]:
# Je vérifie d'abord les valeurs réellement présentes dans la colonne cible.
TARGET = 'fraud_reported'

print("Valeurs uniques :", df_clean[TARGET].unique())
print("\nDistribution :")
print(df_clean[TARGET].value_counts())
print("\nPourcentage :")
print((df_clean[TARGET].value_counts(normalize=True) * 100).round(2))

In [ ]:
# Je convertis la variable cible en 0/1 pour le Machine Learning.
df_clean[TARGET] = df_clean[TARGET].map({'Y': 1, 'N': 0})

fig, ax = plt.subplots(figsize=(7, 5))
counts = df_clean[TARGET].value_counts()
ax.bar(['Non frauduleuse (0)', 'Frauduleuse (1)'],
       counts.values,
       color=['steelblue', 'tomato'],
       edgecolor='white')
for i, v in enumerate(counts.values):
    ax.text(i, v + 5, f'{v} ({v/len(df_clean)*100:.1f}%)',
            ha='center', fontweight='bold')
ax.set_title('Distribution des déclarations (fraude vs non-fraude)', fontweight='bold')
ax.set_ylabel('Nombre de déclarations')
plt.tight_layout()
plt.savefig('figures/distribution_cible.png', dpi=150, bbox_inches='tight')
plt.show()

J'observe ici la répartition entre déclarations frauduleuses et non frauduleuses.
Si les classes sont déséquilibrées, un modèle pourrait obtenir une bonne accuracy
simplement en prédisant toujours la classe majoritaire.
Je vais donc porter une attention particulière au Recall et au ROC-AUC
pour évaluer réellement les performances des modèles.

## Étape 5 — Analyse exploratoire

### Variables numériques

In [ ]:
# Je récupère les variables numériques disponibles (hors cible).
num_cols = df_clean.select_dtypes(include='number').columns.drop(TARGET, errors='ignore').tolist()
print("Variables numériques :", num_cols)

In [ ]:
# Distribution des principales variables numériques.
n = min(len(num_cols), 9)
cols_plot = num_cols[:n]
nrows = (n + 2) // 3

fig, axes = plt.subplots(nrows, 3, figsize=(15, nrows * 4))
axes = axes.ravel()

for i, col in enumerate(cols_plot):
    axes[i].hist(df_clean[col].dropna(), bins=30,
                 color='steelblue', edgecolor='white', alpha=0.85)
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_xlabel('Valeur')
    axes[i].set_ylabel('Fréquence')

for j in range(i+1, len(axes)):
    axes[j].axis('off')

fig.suptitle('Distribution des variables numériques', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('figures/distributions_numeriques.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Je compare les distributions numériques entre fraude et non-fraude avec des boxplots.
n = min(len(num_cols), 6)
cols_box = num_cols[:n]
nrows = (n + 1) // 2

fig, axes = plt.subplots(nrows, 2, figsize=(13, nrows * 4))
axes = axes.ravel()

for i, col in enumerate(cols_box):
    df_clean.boxplot(column=col, by=TARGET, ax=axes[i],
                     boxprops=dict(color='steelblue'),
                     medianprops=dict(color='tomato', linewidth=2))
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_xlabel('Fraude (0 = Non, 1 = Oui)')
    axes[i].set_ylabel('Valeur')

for j in range(i+1, len(axes)):
    axes[j].axis('off')

fig.suptitle('Variables numériques : fraude vs non-fraude', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/boxplots_fraude.png', dpi=150, bbox_inches='tight')
plt.show()

### Variables catégorielles

In [ ]:
# Variables catégorielles disponibles.
cat_cols = df_clean.select_dtypes(include='object').columns.tolist()
print("Variables catégorielles :", cat_cols)

In [ ]:
# Je visualise les 6 variables catégorielles les plus fréquentes.
cat_to_plot = [c for c in cat_cols if df_clean[c].nunique() <= 15][:6]
nrows = (len(cat_to_plot) + 1) // 2

fig, axes = plt.subplots(nrows, 2, figsize=(14, nrows * 4))
axes = axes.ravel()

for i, col in enumerate(cat_to_plot):
    order = df_clean[col].value_counts().index
    sns.countplot(y=df_clean[col], order=order, ax=axes[i],
                  color='steelblue')
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_xlabel('Nombre')
    axes[i].set_ylabel('')

for j in range(i+1, len(axes)):
    axes[j].axis('off')

fig.suptitle('Répartition des variables catégorielles', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/variables_categorielles.png', dpi=150, bbox_inches='tight')
plt.show()

## Étape 6 — Analyse fraude vs non-fraude

Je compare maintenant les déclarations frauduleuses et non frauduleuses
pour identifier les caractéristiques qui semblent davantage associées à la fraude.

In [ ]:
# Je calcule le taux de fraude pour chaque catégorie des variables pertinentes.
fraud_rates = {}
for col in cat_to_plot:
    rate = df_clean.groupby(col)[TARGET].mean().sort_values(ascending=False)
    fraud_rates[col] = rate

# J'affiche les taux de fraude pour les 3 premières variables catégorielles.
n_show = min(3, len(cat_to_plot))
fig, axes = plt.subplots(1, n_show, figsize=(6 * n_show, 5))

if n_show == 1:
    axes = [axes]

for ax, col in zip(axes, cat_to_plot[:n_show]):
    rate = fraud_rates[col]
    colors = ['tomato' if v > rate.mean() else 'steelblue' for v in rate.values]
    rate.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
    ax.axvline(rate.mean(), color='black', linestyle='--', linewidth=1,
               label=f'Moyenne : {rate.mean():.2f}')
    ax.set_title(f'Taux de fraude par {col}', fontweight='bold')
    ax.set_xlabel('Taux de fraude')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('figures/taux_fraude_categories.png', dpi=150, bbox_inches='tight')
plt.show()

# Affichage complet des taux
for col, rate in fraud_rates.items():
    print(f"\nTaux de fraude par {col} :")
    print((rate * 100).round(1).to_string())

Ces taux de fraude par catégorie me permettent d'identifier
quelles modalités sont davantage représentées dans les déclarations frauduleuses.
Je retiens que certaines combinaisons de variables méritent une attention particulière
lors de l'examen manuel des dossiers.

In [ ]:
# Matrice de corrélation entre variables numériques.
corr = df_clean[num_cols + [TARGET]].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax)
ax.set_title('Matrice de corrélation', fontweight='bold')
plt.tight_layout()
plt.savefig('figures/correlation.png', dpi=150, bbox_inches='tight')
plt.show()

J'observe les corrélations entre les variables numériques et la variable cible.
Une corrélation élevée avec la fraude indique une association dans cet échantillon,
mais ne signifie pas qu'une variable cause la fraude.

In [ ]:
# Détection des valeurs extrêmes par boxplot.
# Dans l'assurance, une valeur élevée peut correspondre à un sinistre réel important.
# Je les identifie sans les supprimer automatiquement.
n = min(len(num_cols), 6)
fig, axes = plt.subplots(1, n, figsize=(3 * n, 5))

if n == 1:
    axes = [axes]

for ax, col in zip(axes, num_cols[:n]):
    ax.boxplot(df_clean[col].dropna(), patch_artist=True,
               boxprops=dict(facecolor='steelblue', alpha=0.7),
               medianprops=dict(color='tomato', linewidth=2))
    ax.set_title(col, fontweight='bold', fontsize=8)
    ax.set_xticks([])

fig.suptitle('Détection des valeurs extrêmes', fontweight='bold')
plt.tight_layout()
plt.savefig('figures/outliers.png', dpi=150, bbox_inches='tight')
plt.show()

print("Je garde les valeurs extrêmes : dans le domaine de l'assurance,")
print("un montant élevé peut correspondre à un sinistre réel et important.")

## Étape 7 — Préparation des données pour le Machine Learning

In [ ]:
# Je vérifie qu'aucune variable utilisée comme prédicteur ne contient
# directement l'information de fraude, ce qui constituerait une fuite de données.
# Les variables financières liées aux remboursements déjà effectués
# sont vérifiées manuellement.
print("Colonnes disponibles pour la modélisation :")
print([c for c in df_clean.columns if c != TARGET])

In [ ]:
# Je sépare les variables explicatives de la variable cible.
X = df_clean.drop(columns=[TARGET])
y = df_clean[TARGET]

# Séparation train/test stratifiée pour conserver la proportion de fraudes.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train : {X_train.shape[0]:,} observations")
print(f"Test  : {X_test.shape[0]:,} observations")
print(f"\nProportion fraudes (train) : {y_train.mean():.3f}")
print(f"Proportion fraudes (test)  : {y_test.mean():.3f}")

In [ ]:
# Identification des colonnes numériques et catégorielles dans X.
num_features = X_train.select_dtypes(include='number').columns.tolist()
cat_features = X_train.select_dtypes(include='object').columns.tolist()

print(f"Variables numériques ({len(num_features)}) : {num_features}")
print(f"Variables catégorielles ({len(cat_features)}) : {cat_features}")

In [ ]:
# Pipeline de prétraitement.
# Numériques : imputation par la médiane + standardisation.
# Catégorielles : imputation par la valeur la plus fréquente + One-Hot Encoding.
# Tout est appris sur le train — aucune information du test ne contamine l'entraînement.

num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])

cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ('num', num_pipe, num_features),
    ('cat', cat_pipe, cat_features),
])

## Étape 8 — Modélisation

Je teste trois modèles de classification.

Je commence par la régression logistique comme référence simple,
puis je teste le Random Forest et le Gradient Boosting
pour voir si des modèles plus flexibles font mieux.

Comme les classes peuvent être déséquilibrées, j'utilise `class_weight='balanced'`
pour que le modèle ne néglige pas les déclarations frauduleuses,
qui sont pourtant les plus importantes à détecter.

In [ ]:
# Je définis les trois modèles dans des pipelines complets.
models = {
    'Logistic Regression': Pipeline([
        ('prep',  preprocessor),
        ('model', LogisticRegression(
            class_weight='balanced', max_iter=1000, random_state=42)),
    ]),
    'Random Forest': Pipeline([
        ('prep',  preprocessor),
        ('model', RandomForestClassifier(
            n_estimators=100, class_weight='balanced',
            random_state=42, n_jobs=-1)),
    ]),
    'Gradient Boosting': Pipeline([
        ('prep',  preprocessor),
        ('model', GradientBoostingClassifier(
            n_estimators=100, random_state=42)),
    ]),
}

# Entraînement de tous les modèles.
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    print(f"{name} : entraîné.")

## Étape 9 — Évaluation des modèles

In [ ]:
# Je calcule les métriques pour chaque modèle sur le jeu de test.
results = []

for name, pipe in models.items():
    y_pred      = pipe.predict(X_test)
    y_pred_prob = pipe.predict_proba(X_test)[:, 1]

    results.append({
        'Modèle'   : name,
        'Accuracy' : accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall'   : recall_score(y_test, y_pred, zero_division=0),
        'F1-score' : f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC'  : roc_auc_score(y_test, y_pred_prob),
    })

results_df = pd.DataFrame(results).set_index('Modèle').round(3)
print(results_df)

### Lecture des métriques

**Accuracy** — proportion globale de prédictions correctes.

**Precision** — parmi les déclarations que le modèle identifie comme frauduleuses, combien le sont réellement ?

**Recall** — parmi les déclarations réellement frauduleuses, combien le modèle arrive-t-il à détecter ?

**F1-score** — compromis entre précision et rappel.

**ROC-AUC** — capacité générale du modèle à distinguer les deux classes.

Dans ce problème, le **Recall est particulièrement important** : une fraude non détectée représente un coût direct pour la compagnie. Mais augmenter le recall au détriment de la précision revient à signaler trop de déclarations normales comme suspectes, ce qui surcharge les équipes de vérification. Il faut donc trouver un bon équilibre.

In [ ]:
# Courbes ROC pour les trois modèles.
fig, ax = plt.subplots(figsize=(8, 6))

colors = ['steelblue', 'darkorange', 'seagreen']

for (name, pipe), color in zip(models.items(), colors):
    y_prob = pipe.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})', color=color, linewidth=2)

ax.plot([0,1],[0,1], 'k--', linewidth=1, label='Aléatoire (AUC = 0.5)')
ax.set_xlabel('Taux de faux positifs')
ax.set_ylabel('Taux de vrais positifs (Recall)')
ax.set_title('Courbes ROC — comparaison des modèles', fontweight='bold')
ax.legend(loc='lower right')

plt.tight_layout()
plt.savefig('figures/courbes_roc.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Matrice de confusion pour le meilleur modèle selon le ROC-AUC.
best_name = results_df['ROC-AUC'].idxmax()
best_pipe = models[best_name]
y_pred_best = best_pipe.predict(X_test)

cm = confusion_matrix(y_test, y_pred_best)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Prédit Non-Fraude', 'Prédit Fraude'],
            yticklabels=['Réel Non-Fraude', 'Réel Fraude'], ax=ax)
ax.set_title(f'Matrice de confusion — {best_name}', fontweight='bold')

plt.tight_layout()
plt.savefig('figures/matrice_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"Vrais négatifs  (TN) : {tn} — déclarations normales correctement identifiées")
print(f"Faux positifs   (FP) : {fp} — déclarations normales signalées à tort comme frauduleuses")
print(f"Faux négatifs   (FN) : {fn} — fraudes passées inaperçues")
print(f"Vrais positifs  (TP) : {tp} — fraudes correctement détectées")

Un faux négatif correspond à une déclaration réellement frauduleuse
que le modèle a classée comme normale. C'est le cas le plus coûteux
pour la compagnie d'assurance.

Un faux positif correspond à une déclaration normale signalée à tort.
Cela représente un surcoût de traitement mais n'entraîne pas de perte financière directe.

In [ ]:
# Importance des variables pour le meilleur modèle.
model_step = best_pipe.named_steps['model']
prep_step  = best_pipe.named_steps['prep']

if hasattr(model_step, 'feature_importances_'):
    importances = model_step.feature_importances_
    method = 'feature_importances_'
elif hasattr(model_step, 'coef_'):
    importances = np.abs(model_step.coef_[0])
    method = 'coef_'
else:
    importances = None

if importances is not None:
    try:
        feature_names = prep_step.get_feature_names_out()
    except Exception:
        feature_names = [f'feature_{i}' for i in range(len(importances))]

    imp_series = pd.Series(importances, index=feature_names)
    imp_top    = imp_series.sort_values(ascending=False).head(15)

    fig, ax = plt.subplots(figsize=(10, 6))
    imp_top.sort_values().plot(kind='barh', ax=ax,
                               color='steelblue', edgecolor='white')
    ax.set_title(f'Top 15 variables importantes — {best_name}', fontweight='bold')
    ax.set_xlabel('Importance')

    plt.tight_layout()
    plt.savefig('figures/importance_variables.png', dpi=150, bbox_inches='tight')
    plt.show()

J'observe que certaines variables contribuent davantage aux décisions du modèle.
Cela ne signifie pas qu'elles causent la fraude, mais qu'elles jouent
un rôle important dans la discrimination entre les deux classes.
Ces variables méritent une attention particulière lors de l'examen des dossiers.

## Étape 10 — Interprétation métier et recommandations

### Ce que les résultats peuvent apporter à une compagnie d'assurance

Le modèle construit ici n'accuse pas un client de fraude.
Il produit un score de risque pour chaque déclaration.
Ce score peut être utilisé pour :

- identifier automatiquement les déclarations qui méritent une vérification manuelle approfondie ;
- prioriser les dossiers à examiner en commençant par ceux avec le score le plus élevé ;
- réduire le temps consacré à l'analyse de déclarations manifestement normales ;
- concentrer les ressources des équipes de contrôle sur les cas les plus suspects.

### Recommandations

Je propose les recommandations suivantes à partir de mes résultats.

Mettre en place un système de scoring automatique des nouvelles déclarations,
basé sur le modèle entraîné ici.

Examiner manuellement les déclarations dont le score de fraude dépasse un seuil
à définir selon le compromis souhaité entre Recall et Precision.

Ne pas utiliser ce modèle seul comme unique critère de décision :
il doit rester un outil d'aide à la décision, pas un système automatique d'accusation.

Réentraîner régulièrement le modèle avec de nouvelles données
car les schémas de fraude évoluent dans le temps.

## Limites de l'analyse

Ce projet présente plusieurs limites importantes à garder à l'esprit.

Le dataset est de taille modeste. Les performances obtenues
pourraient être différentes sur un portefeuille réel de plusieurs millions de contrats.

Les associations identifiées entre certaines variables et la fraude
sont des corrélations statistiques dans cet échantillon.
Elles ne permettent pas de conclure à une relation causale.

Le déséquilibre éventuel des classes rend l'évaluation délicate.
L'accuracy seule n'est pas un indicateur suffisant.

Un modèle entraîné sur ce dataset ne peut pas être automatiquement
transposé à une autre compagnie d'assurance sans réévaluation.

Enfin, prédire un risque de fraude ne signifie pas qu'une déclaration est frauduleuse :
le modèle produit une probabilité, pas une certitude.

## Conclusion

Dans ce projet, j'ai exploré un dataset de déclarations de sinistres automobiles
pour analyser les caractéristiques associées à la fraude
et construire des modèles capables de la détecter.

L'analyse exploratoire m'a permis d'identifier des variables
qui présentent des distributions différentes entre les déclarations
frauduleuses et non frauduleuses.

Les trois modèles testés montrent des performances différentes.
Le meilleur modèle selon le ROC-AUC est le suivant :

In [ ]:
# Résumé des performances finales.
print("Comparaison des modèles (sur le jeu de test) :")
print(results_df.to_string())
print(f"\nMeilleur modèle (ROC-AUC) : {best_name}")

Ce projet m'a appris à travailler sur un problème de classification déséquilibrée,
à choisir les métriques adaptées au contexte métier,
et à interpréter les résultats dans une perspective concrète pour la compagnie d'assurance.

Il complète mes deux projets précédents en couvrant un aspect différent de l'actuariat :
la détection du risque de fraude, qui est un enjeu central pour les assureurs.